# X-OPT - REOPTIMIZATION WITH INTERPRETABILITY INSIGHTS

This notebook evaluates reoptimization over the OR-Library p-median instances by combining the existing metaheuristic pipeline with post-hoc structural interpretability. For each instance, the workflow is:

1. solve the original instance with the metaheuristic and record runtime, best value, and gap to the published optimum;
2. extract the max-p-cut, highest k-core, densest subgraph, and the union between the highest k-core and the densest subgraph from the long-term-memory co-occurrence graph;
3. perturb the original graph edge costs and recompute the shortest-path distance matrix;
4. reoptimize the perturbed instance with five `python-mip` variants:
   - unrestricted p-median;
   - p-median with exactly one facility selected from each Max-p-Cut partition;
   - p-median restricted to facilities in the highest k-core;
   - p-median restricted to facilities in the densest subgraph;
   - p-median restricted to facilities in the union between the highest k-core and the densest subgraph.

The perturbation is a symmetric multiplicative noise on the original edge costs, followed by shortest-path recomputation so the perturbed instance remains graph-based.

### SETUP

In [1]:
from __future__ import annotations

import gc
import os
import sys

import numpy  as np
import pandas as pd

from pathlib         import Path
from tqdm.auto       import tqdm
from IPython.display import display
from time            import perf_counter
from mip             import BINARY, Model, OptimizationStatus, minimize, xsum


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

In [2]:
from lib.paths     import find_project_root, \
                          instance_sort_key, \
                          canonical_instance_name
from lib.instances import list_orlibrary_instances, \
                          apply_instance_selection, \
                          read_instance_metadata  , \
                          load_best_known_costs_to_dict_id

from lib.graph     import read_orlibrary_graph, \
                          all_pairs_shortest_paths
from lib.mip       import extract_open_facilities_candidates

from lib.metrics   import compute_solution_cost   , \
                          gap_to_reference_percent, \
                          improvement_percent
from lib.explain   import extract_structure_insights

from lib.utils     import parse_optional_int_env  , \
                          parse_optional_float_env, \
                          finite_or_none

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


In [4]:
import pymedian

### CONFIGURATION

In [5]:
INSTANCES_DIR = PROJECT_ROOT  / 'instances'
PMEDOPT_PATH  = INSTANCES_DIR / 'pmedopt.txt'
OUTPUT_DIR    = PROJECT_ROOT  / 'notebooks' / 'experiments_sbpo' / 'artifacts'

RAW_RESULTS_CSV       = OUTPUT_DIR / 'reoptimization_interpretability_raw.csv'
SUMMARY_RESULTS_CSV   = OUTPUT_DIR / 'reoptimization_interpretability_summary.csv'
VARIANT_AGGREGATE_CSV = OUTPUT_DIR / 'reoptimization_interpretability_variant_aggregate.csv'
FAILURES_CSV          = OUTPUT_DIR / 'reoptimization_interpretability_failures.csv'

INSTANCE_FILTER = os.getenv             ('REOPT_INSTANCE_FILTER')
MAX_INSTANCES   = parse_optional_int_env('REOPT_MAX_INSTANCES'  ) or 20

RESTARTS       = parse_optional_int_env  ('REOPT_META_RESTARTS') or 8
MAX_ITER       = parse_optional_int_env  ('REOPT_META_MAX_ITER') or 25
FACTOR         = parse_optional_int_env  ('REOPT_META_FACTOR'  ) or 1
TOP_FRACTION   = parse_optional_float_env('REOPT_TOP_FRACTION' ) or 0.10
DETAILS_FORMAT = os.getenv    ('REOPT_DETAILS_FORMAT', 'binary')

TIME_LIMIT_SECONDS  = parse_optional_int_env  ('REOPT_TIME_LIMIT_SECONDS') or 400
PERTURBATION_DELTA  = parse_optional_float_env('REOPT_PERTURBATION_DELTA') or 0.1
GLOBAL_SEED         = parse_optional_int_env  ('REOPT_GLOBAL_SEED'       ) or 42
MAX_P_CUT_RESTARTS  = parse_optional_int_env  ('REOPT_MAX_P_CUT_RESTARTS') or 10
MAX_P_CUT_MAX_ITER  = parse_optional_int_env  ('REOPT_MAX_P_CUT_MAX_ITER') or 1000

SAVE_RESULTS_CSV = os.getenv('REOPT_SAVE_RESULTS_CSV', '1') != '0'

ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)

BEST_KNOWN_COSTS = load_best_known_costs_to_dict_id(PMEDOPT_PATH)


if not INSTANCE_NAMES:
    raise FileNotFoundError(
        f'No instances were selected from {INSTANCES_DIR}.'
    )

In [6]:
print(f'Instances discovered : {len(ALL_INSTANCE_NAMES)}')
print(f'Instances selected   : {len(INSTANCE_NAMES    )}')

print()

print(f'Metaheuristic parameters       : restarts={RESTARTS          :2d}, max_iter={MAX_ITER          :4d}, factor={FACTOR     }')
print(f'Max-p-Cut parameters           : restarts={MAX_P_CUT_RESTARTS:2d}, max_iter={MAX_P_CUT_MAX_ITER:4d}, seed  ={GLOBAL_SEED}')
print(f'Top fraction kept from the LTM : {TOP_FRACTION:.0%}')

Instances discovered : 40
Instances selected   : 20

Metaheuristic parameters       : restarts= 8, max_iter=  25, factor=1
Max-p-Cut parameters           : restarts=10, max_iter=1000, seed  =42
Top fraction kept from the LTM : 10%


In [7]:
print(f'Perturbation delta            : {PERTURBATION_DELTA:.1%}')
print(f'Reoptimization time limit (s) : {TIME_LIMIT_SECONDS    }')

print(f'Raw results CSV       : {RAW_RESULTS_CSV      }')
print(f'Summary results CSV   : {SUMMARY_RESULTS_CSV  }')
print(f'Variant aggregate CSV : {VARIANT_AGGREGATE_CSV}')

Perturbation delta            : 10.0%
Reoptimization time limit (s) : 400
Raw results CSV       : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_interpretability_raw.csv
Summary results CSV   : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_interpretability_summary.csv
Variant aggregate CSV : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_interpretability_variant_aggregate.csv


### PERTURBATION

In [8]:
def perturb_adjacency(
    adjacency: list[list[tuple[int, int]]],
    *,
    delta             : float,
    seed              : int  ,
    affected_fraction : float = 0.10,
) -> tuple[list[list[tuple[int, int]]], dict[str, float | int]]:
    rng = np.random.default_rng(seed)
    n   = len(adjacency)

    perturbed = [[] for _ in range(n)]

    unique_edges = []
    for u, neighbors in enumerate(adjacency):
        for v, weight in neighbors:
            if u < v:
                unique_edges.append((u, v, weight))

    edge_count     = len(unique_edges)
    affected_count = max(1, int(round(edge_count * affected_fraction)))

    affected_idx = set(
        rng.choice(
            edge_count, size=affected_count, replace=False
        )
    )

    multipliers = []

    for idx, (u, v, weight) in enumerate(unique_edges):
        if idx in affected_idx:
            multiplier = 1.0 + float(rng.uniform(-delta, delta))
            new_weight = max(1, int(round(weight * multiplier)))

            multipliers.append(multiplier)
        else:
            multiplier = 1.0
            new_weight = weight

        perturbed[u].append((v, new_weight))
        perturbed[v].append((u, new_weight))

    for neighbors in perturbed:
        neighbors.sort()

    multipliers_array = np.asarray(multipliers, dtype=float)

    stats = {
        'perturbation_edge_count'          : edge_count    ,
        'perturbation_affected_edge_count' : affected_count,
        'perturbation_affected_fraction'   : affected_count / edge_count     if edge_count             else 0.0,
        'perturbation_mean_multiplier'     : float(multipliers_array.mean()) if len(multipliers_array) else 1.0,
    }

    return perturbed, stats

### EXACT REOPTIMIZATION

In [9]:
def build_pmedian_model(
    distances         : np.ndarray,
    p                 : int       ,
    *,
    allowed_facilities: list[int] | tuple[int, ...] | None = None,
    partition_groups  : list[tuple[int, ...]      ] | None = None,
) -> tuple[Model, list[list], list, list[int]]:
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError('Distances must be a square 2D array.')

    n = distances.shape[0]
    if not (1 <= p <= n):
        raise ValueError('P must satisfy 1 <= p <= n.')

    if allowed_facilities is None:
        candidate_facilities = list(range(n))
    else:
        candidate_facilities = sorted(
            {
                int(facility)
                for facility in allowed_facilities
            }
        )

    if len(candidate_facilities) < p:
        raise ValueError(
            f'Candidate facility pool is too small: {len(candidate_facilities)} < {p}.'
        )

    model = Model(solver_name='CBC')
    model.verbose = 0

    y             = [
        model.add_var(
            var_type=BINARY
        )
        for _ in candidate_facilities
    ]
    x: list[list] = []

    for i in range(n):
        x_row = [
            model.add_var(
                var_type=BINARY
            )
            for _ in candidate_facilities
        ]
        x.append(x_row)

        model.add_constr(xsum(x_row) == 1)

        for pos in range(len(candidate_facilities)):
            model.add_constr(x_row[pos] <= y[pos])

    model.add_constr(xsum(y) == p)

    if partition_groups is not None:
        if len(partition_groups) != p:
            raise ValueError(
                f'Expected {p} partition groups, but received {len(partition_groups)}.'
            )

        facility_to_pos = {
            facility: pos
            for pos, facility in enumerate(candidate_facilities)
        }

        for group_nodes in partition_groups:
            group_positions = [
                facility_to_pos[node]
                for node in group_nodes
                if  node in facility_to_pos
            ]

            if not group_positions:
                raise ValueError(
                    'A Max-p-Cut partition has no available candidate facility.'
                )

            model.add_constr(xsum(y[pos] for pos in group_positions) == 1)

    model.objective = minimize(
        xsum(
            float(distances[i, facility]) * x[i][pos]
            for i in range(n)
            for pos, facility in enumerate(candidate_facilities)
        )
    )

    return model, x, y, candidate_facilities


def optimize_model(
    model              : Model     ,
    time_limit_seconds : int | None,
) -> OptimizationStatus:
    if time_limit_seconds is None:
        return model.optimize()

    return model.optimize(max_seconds=float(time_limit_seconds))


def solve_reoptimization_variant(
    *,
    variant            : str       ,
    variant_order      : int       ,
    distances          : np.ndarray,
    p                  : int       ,
    time_limit_seconds : int | None,
    allowed_facilities : list[int] | tuple[int, ...] | None = None,
    partition_groups   : list[tuple[int, ...]      ] | None = None,
) -> dict[str, object]:
    n = distances.shape[0]

    candidate_facilities = (
        list(range(n))
        if   allowed_facilities is None
        else sorted(
            {
                int(value)
                for value in allowed_facilities
            }
        )
    )

    row = {
        'variant'                     : variant      ,
        'variant_order'               : variant_order,
        'candidate_facility_count'    : len(candidate_facilities)            ,
        'candidate_facility_fraction' : len(candidate_facilities) / max(1, n),
        'partition_group_count'       : len(partition_groups    ) if partition_groups is not None else np.nan,

        'objective_value'       : np.nan,
        'model_build_seconds'   : np.nan,
        'solve_seconds'         : np.nan,
        'total_variant_seconds' : np.nan,

        'open_facilities_count' : 0 ,
        'open_facilities'       : '',

        'status'        : 'ERROR',
        'error_message' : None   ,
    }

    if len(candidate_facilities) < p:
        row.update(
            {
                'status'        : 'INFEASIBLE_POOL',
                'error_message' : f'Candidate facility pool smaller than p: {len(candidate_facilities)} < {p}.',
            }
        )

        return row

    build_started_at = perf_counter()

    try:
        model, _, y, candidate_facilities = build_pmedian_model(
            distances,
            p        ,
            allowed_facilities=candidate_facilities,
            partition_groups  =partition_groups    ,
        )
        build_seconds = perf_counter() - build_started_at

        solve_started_at = perf_counter  ()
        status           = optimize_model(model, time_limit_seconds)
        solve_seconds    = perf_counter  () - solve_started_at

        has_incumbent = status in {
            OptimizationStatus.OPTIMAL ,
            OptimizationStatus.FEASIBLE,
        }

        solver_objective = finite_or_none(
            model.objective_value if has_incumbent else None
        )

        open_facilities: list[int] = []
        validated_objective        = None

        if has_incumbent:
            open_facilities = extract_open_facilities_candidates(
                candidate_facilities, y
            )

            if len(open_facilities) != p:
                raise ValueError(
                    f'Expected {p} open facilities, but recovered {len(open_facilities)}.'
                )

            validated_objective = compute_solution_cost(
                distances, open_facilities
            )

            if solver_objective    is not None and \
               validated_objective is not None:
                if abs(solver_objective - validated_objective) > 1e-4:
                    raise ValueError(
                         'Solver objective and validated objective do not match: '
                        f'{solver_objective} vs {validated_objective}.'
                    )

        objective_value = (
            validated_objective
            if   validated_objective is not None
            else solver_objective
        )

        row.update(
            {
                'objective_value'       : objective_value if objective_value is not None else np.nan,
                'model_build_seconds'   : build_seconds                ,
                'solve_seconds'         : solve_seconds                ,
                'total_variant_seconds' : build_seconds + solve_seconds,

                'open_facilities_count' : len     (open_facilities),
                'open_facilities'       : ' '.join(
                    str(facility + 1) for facility in open_facilities
                ),

                'status'        : getattr(status, 'name', str(status)),
                'error_message' : None                                ,
            }
        )
    except Exception as exc:
        row.update(
            {
                'status'              : 'ERROR',
                'error_message'       : f'{type(exc).__name__}: {exc}'   ,
                'model_build_seconds' : perf_counter() - build_started_at,
            }
        )
    finally:
        for name in ['model', 'x', 'y']:
            if name in locals():
                del locals()[name]

        gc.collect()

    return row

### BENCHMARK EXECUTION

In [10]:
VARIANT_SPECS = [
    {
        'variant'      : 'baseline',
        'variant_order': 1   ,
        'allowed_key'  : None,
        'groups_key'   : None,
    },
    {
        'variant'      : 'max_p_cut',
        'variant_order': 2   ,
        'allowed_key'  : None,
        'groups_key'   : 'max_p_cut_groups',
    },
    {
        'variant'      : 'highest_k_core'      ,
        'variant_order': 3                     ,
        'allowed_key'  : 'highest_k_core_nodes',
        'groups_key'   : None,
    },
    {
        'variant'      : 'densest_subgraph',
        'variant_order': 4                 ,
        'allowed_key'  : 'densest_nodes'   ,
        'groups_key'   : None,
    },
    {
        'variant'      : 'k_core_densest_union'      ,
        'variant_order': 5                           ,
        'allowed_key'  : 'k_core_densest_union_nodes',
        'groups_key'   : None,
    },
]


In [11]:
def build_pipeline_error_rows(
    instance_name   : str,
    instance_id     : str,
    metadata        : dict[str, int],
    best_known_cost : float | None  ,
    error_message   : str           ,
) -> list[dict[str, object]]:
    rows = []

    for spec in VARIANT_SPECS:
        rows.append(
            {
                'instance'                    : instance_name,
                'instance_id'                 : instance_id  ,
                'n'                           : metadata.get('n', np.nan),
                'p'                           : metadata.get('p', np.nan),
                'best_known_cost'             : best_known_cost if best_known_cost is not None else np.nan,

                'metaheuristic_best_cost'               : np.nan,
                'metaheuristic_gap_percent'             : np.nan,
                'metaheuristic_runtime_seconds'         : np.nan,
                'structure_extraction_runtime_seconds'  : np.nan,
                'perturbation_runtime_seconds'          : np.nan,
                'pre_reoptimization_seconds'            : np.nan,
                'previous_solution_perturbed_objective' : np.nan,
                'previous_solution_degradation_percent' : np.nan,
                'memory_size'                           : np.nan,
                'top_solution_count'                    : np.nan,
                'top_cost_cutoff'                       : np.nan,
                'cooccurrence_edges'                    : np.nan,
                'cooccurrence_total_weight'             : np.nan,

                'max_p_cut_fraction_cut'              : np.nan,
                'max_p_cut_best_facility_separation'  : np.nan,
                'k_core_max_level'                    : np.nan,
                'k_core_candidate_count'              : np.nan,
                'k_core_candidate_fraction'           : np.nan,
                'k_core_best_set_recall'              : np.nan,
                'densest_subgraph_candidate_count'      : np.nan,
                'densest_subgraph_candidate_fraction'   : np.nan,
                'densest_subgraph_density'              : np.nan,
                'densest_subgraph_best_set_recall'      : np.nan,
                'k_core_densest_union_candidate_count'  : np.nan,
                'k_core_densest_union_candidate_fraction': np.nan,
                'k_core_densest_union_best_set_recall'  : np.nan,

                'perturbation_delta'                    : PERTURBATION_DELTA,
                'perturbation_seed'                     : np.nan,
                'perturbation_edge_count'               : np.nan,
                'perturbation_mean_multiplier'          : np.nan,

                'variant'                     : spec['variant'      ],
                'variant_order'               : spec['variant_order'],
                'candidate_facility_count'    : np.nan,
                'candidate_facility_fraction' : np.nan,
                'partition_group_count'       : np.nan,

                'objective_value'                   : np.nan,
                'improvement_over_previous_percent' : np.nan,
                'gap_to_unrestricted_percent'       : np.nan,
                'model_build_seconds'               : np.nan,
                'solve_seconds'                     : np.nan,
                'total_variant_seconds'             : np.nan,
                'full_pipeline_seconds'             : np.nan,
                'open_facilities_count'             : 0 ,
                'open_facilities'                   : '',

                'status'        : 'PIPELINE_ERROR',
                'error_message' : error_message   ,
            }
        )

    return rows

In [12]:
def run_single_instance_analysis(
    instance_name     : str,
    *,
    perturbation_seed : int  ,
    restarts          : int  ,
    max_iter          : int  ,
    factor            : int  ,
    top_fraction      : float,
    details_format    : str  ,
    max_p_cut_restarts: int  ,
    max_p_cut_max_iter: int  ,
    global_seed       : int  ,
    perturbation_delta: float,
    time_limit_seconds: int | None,
) -> list[dict[str, object]]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    metadata        = read_instance_metadata(instance_path)
    best_known_cost = BEST_KNOWN_COSTS.get  (instance_id  )

    try:
        meta_started_at  = perf_counter()
        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =restarts      ,
            max_iter      =max_iter      ,
            factor        =factor        ,
            details_format=details_format,
        )
        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        structure_started_at = perf_counter()
        insights             = extract_structure_insights(
            summary,
            details,
            top_fraction       =top_fraction      ,
            max_p_cut_restarts =max_p_cut_restarts,
            max_p_cut_max_iter =max_p_cut_max_iter,
            global_seed        =global_seed       ,
        )
        structure_runtime_seconds = perf_counter() - structure_started_at

        perturbation_started_at = perf_counter()

        original_graph                          = read_orlibrary_graph(instance_path)
        perturbed_adjacency, perturbation_stats = perturb_adjacency   (
            original_graph['adjacency'],
            delta=perturbation_delta,
            seed =perturbation_seed ,
        )
        perturbed_distances          = all_pairs_shortest_paths(perturbed_adjacency)
        perturbation_runtime_seconds = perf_counter() - perturbation_started_at

        previous_solution_perturbed_objective = compute_solution_cost(
            perturbed_distances, list(insights['best_facilities'])
        )

        common = {
            'instance'                     : instance_name,
            'instance_id'                  : instance_id  ,
            'n'                            : int(summary['n']),
            'p'                            : int(summary['p']),
            'best_known_cost'              : best_known_cost if best_known_cost is not None else np.nan,

            'metaheuristic_best_cost'       : insights['best_cost']        ,
            'metaheuristic_runtime_seconds' : metaheuristic_runtime_seconds,
            'metaheuristic_gap_percent'     : gap_to_reference_percent(insights['best_cost'], best_known_cost),

            'structure_extraction_runtime_seconds'  : structure_runtime_seconds   ,
            'perturbation_runtime_seconds'          : perturbation_runtime_seconds,
            'pre_reoptimization_seconds'            : metaheuristic_runtime_seconds + structure_runtime_seconds + perturbation_runtime_seconds,
            'previous_solution_degradation_percent' : gap_to_reference_percent(previous_solution_perturbed_objective, insights['best_cost']               ),
            'previous_solution_perturbed_objective' : previous_solution_perturbed_objective if previous_solution_perturbed_objective is not None else np.nan,

            'memory_size'               : insights['memory_size'              ],
            'top_solution_count'        : insights['top_solution_count'       ],
            'top_cost_cutoff'           : insights['top_cost_cutoff'          ],
            'cooccurrence_edges'        : insights['cooccurrence_edges'       ],
            'cooccurrence_total_weight' : insights['cooccurrence_total_weight'],

            'max_p_cut_fraction_cut'             : insights['max_p_cut_fraction_cut'            ],
            'max_p_cut_best_facility_separation' : insights['max_p_cut_best_facility_separation'],

            'k_core_max_level'          : insights['k_core_max_level'         ],
            'k_core_candidate_count'    : insights['k_core_candidate_count'   ],
            'k_core_candidate_fraction' : insights['k_core_candidate_fraction'],
            'k_core_best_set_recall'    : insights['k_core_best_set_recall'   ],

            'densest_subgraph_candidate_count'      : insights['densest_subgraph_candidate_count'     ],
            'densest_subgraph_candidate_fraction'   : insights['densest_subgraph_candidate_fraction'  ],
            'densest_subgraph_density'              : insights['densest_subgraph_density'             ],
            'densest_subgraph_best_set_recall'      : insights['densest_subgraph_best_set_recall'     ],
            'k_core_densest_union_candidate_count'  : insights['k_core_densest_union_candidate_count' ],
            'k_core_densest_union_candidate_fraction': insights['k_core_densest_union_candidate_fraction'],
            'k_core_densest_union_best_set_recall'  : insights['k_core_densest_union_best_set_recall' ],

            'perturbation_delta' : perturbation_delta,
            'perturbation_seed'  : perturbation_seed ,
            **perturbation_stats,
        }

        rows = []

        for spec in VARIANT_SPECS:
            allowed_facilities = None
            if spec['allowed_key'] is not None:
                allowed_facilities = sorted(insights[spec['allowed_key']])

            partition_groups = None
            if spec['groups_key'] is not None:
                partition_groups = insights[spec['groups_key']]

            row = solve_reoptimization_variant(
                variant            =spec['variant'      ],
                variant_order      =spec['variant_order'],
                distances          =perturbed_distances  ,
                p                  =int(summary['p']    ),
                time_limit_seconds =time_limit_seconds,
                allowed_facilities =allowed_facilities,
                partition_groups   =partition_groups  ,
            )

            row.update(common)

            row['improvement_over_previous_percent'] = improvement_percent(
                previous_solution_perturbed_objective,
                row['objective_value'] if pd.notna(row['objective_value']) else None,
            )

            row['full_pipeline_seconds'] = common['pre_reoptimization_seconds'] + (
                row['total_variant_seconds'] if pd.notna(row['total_variant_seconds']) else 0.0
            )

            rows.append(row)

        return rows
    except Exception as exc:
        return build_pipeline_error_rows(
            instance_name  ,
            instance_id    ,
            metadata       ,
            best_known_cost,
            f'{type(exc).__name__}: {exc}',
        )


def run_benchmark(
    instance_names: list[str],
    *,
    restarts          : int  ,
    max_iter          : int  ,
    factor            : int  ,
    top_fraction      : float,
    details_format    : str  ,
    max_p_cut_restarts: int  ,
    max_p_cut_max_iter: int  ,
    global_seed       : int  ,
    perturbation_delta: float,
    time_limit_seconds: int | None,
) -> pd.DataFrame:
    if not instance_names:
        raise ValueError('The benchmark requires at least one instance.')

    rows = []
    for offset, instance_name in enumerate(
        tqdm(
            instance_names,
            total        =len(instance_names)       ,
            desc         ='Reoptimization benchmark',
            dynamic_ncols=True                      ,
        )
    ):
        rows.extend(
            run_single_instance_analysis(
                instance_name,
                perturbation_seed =global_seed + offset,
                restarts          =restarts            ,
                max_iter          =max_iter            ,
                factor            =factor              ,
                top_fraction      =top_fraction        ,
                details_format    =details_format      ,
                max_p_cut_restarts=max_p_cut_restarts  ,
                max_p_cut_max_iter=max_p_cut_max_iter  ,
                global_seed       =global_seed         ,
                perturbation_delta=perturbation_delta  ,
                time_limit_seconds=time_limit_seconds  ,
            )
        )

    results_df = pd.DataFrame(rows)

    results_df['instance_order'] = results_df['instance'].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        results_df.sort_values(['instance_order', 'instance', 'variant_order'])
                  .drop       (columns='instance_order')
                  .reset_index(drop   =True            )
    )


def add_unrestricted_reference(results_df: pd.DataFrame) -> pd.DataFrame:
    baseline_df = (
        results_df.loc[results_df['variant'] == 'baseline', ['instance', 'status', 'objective_value']]
                  .rename(
                      columns={
                          'status'         : 'baseline_status'         ,
                          'objective_value': 'baseline_objective_value',
                      }
                  )
    )

    merged = results_df.merge(baseline_df, on='instance', how='left')

    merged['gap_to_unrestricted_percent'] = np.where(
        merged['objective_value'].notna() & merged['baseline_objective_value'].notna(),
        100.0 * (merged['objective_value'] - merged['baseline_objective_value']) / merged['baseline_objective_value'],
        np.nan,
    )

    merged['baseline_is_optimal'] = merged['baseline_status'].eq('OPTIMAL')

    return merged


def build_wide_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    instance_columns = [
        'instance'       ,
        'instance_id'    ,
        'n'              ,
        'p'              ,
        'best_known_cost',

        'metaheuristic_best_cost'              ,
        'metaheuristic_gap_percent'            ,
        'metaheuristic_runtime_seconds'        ,
        'structure_extraction_runtime_seconds' ,
        'perturbation_runtime_seconds'         ,
        'pre_reoptimization_seconds'           ,
        'previous_solution_perturbed_objective',
        'previous_solution_degradation_percent',

        'memory_size'              ,
        'top_solution_count'       ,
        'top_cost_cutoff'          ,
        'cooccurrence_edges'       ,
        'cooccurrence_total_weight',

        'max_p_cut_fraction_cut'            ,
        'max_p_cut_best_facility_separation',

        'k_core_max_level'         ,
        'k_core_candidate_count'   ,
        'k_core_candidate_fraction',
        'k_core_best_set_recall'   ,

        'densest_subgraph_candidate_count'    ,
        'densest_subgraph_candidate_fraction' ,
        'densest_subgraph_density'            ,
        'densest_subgraph_best_set_recall'    ,
        'k_core_densest_union_candidate_count',
        'k_core_densest_union_candidate_fraction',
        'k_core_densest_union_best_set_recall',

        'perturbation_delta'                   ,
        'perturbation_seed'                    ,
        'perturbation_edge_count'              ,
        'perturbation_mean_multiplier'         ,
    ]

    metric_columns = [
        'status'                     ,
        'candidate_facility_count'   ,
        'candidate_facility_fraction',

        'objective_value'   ,

        'improvement_over_previous_percent',
        'gap_to_unrestricted_percent'      ,

        'model_build_seconds'  ,
        'solve_seconds'        ,
        'total_variant_seconds',
        'full_pipeline_seconds',
    ]

    base_df = results_df[instance_columns].drop_duplicates('instance')

    wide_df         = results_df.pivot(index='instance', columns='variant', values=metric_columns)
    wide_df.columns = [f'{variant}__{metric}' for metric, variant in wide_df.columns]
    wide_df         = wide_df.reset_index()

    return base_df.merge(wide_df, on='instance', how='left')


def build_variant_aggregate(results_df: pd.DataFrame) -> pd.DataFrame:
    return (
        results_df.groupby('variant', dropna=False)
                  .agg(
                      instances                         =('instance', 'nunique'),
                      solved_instances                  =('objective_value', lambda s: int( s.notna()      .sum())),
                      optimal_instances                 =('status'         , lambda s: int((s == 'OPTIMAL').sum())),
                      mean_candidate_facility_fraction  =('candidate_facility_fraction'      , 'mean'),
                      mean_total_variant_seconds        =('total_variant_seconds'            , 'mean'),
                      mean_full_pipeline_seconds        =('full_pipeline_seconds'            , 'mean'),
                      mean_improvement_over_previous_pct=('improvement_over_previous_percent', 'mean'),
                      mean_gap_to_unrestricted_pct      =('gap_to_unrestricted_percent'      , 'mean'),
                  )
                  .reset_index()
                  .sort_values('variant')
                  .reset_index(drop=True)
    )

### APPLY

In [13]:
raw_results_df = run_benchmark(
    INSTANCE_NAMES,
    restarts          =RESTARTS          ,
    max_iter          =MAX_ITER          ,
    factor            =FACTOR            ,
    top_fraction      =TOP_FRACTION      ,
    details_format    =DETAILS_FORMAT    ,
    max_p_cut_restarts=MAX_P_CUT_RESTARTS,
    max_p_cut_max_iter=MAX_P_CUT_MAX_ITER,
    global_seed       =GLOBAL_SEED       ,
    perturbation_delta=PERTURBATION_DELTA,
    time_limit_seconds=TIME_LIMIT_SECONDS,
)

Reoptimization benchmark:   0%|          | 0/20 [00:00<?, ?it/s]

In [14]:
results_df           = add_unrestricted_reference(raw_results_df)
wide_summary_df      = build_wide_summary        (results_df)
variant_aggregate_df = build_variant_aggregate   (results_df)
failures_df          = results_df[results_df['error_message'].notna()].copy()

In [15]:
display(variant_aggregate_df.round(4))

,variant,instances,solved_instances,optimal_instances,mean_candidate_facility_fraction,mean_total_variant_seconds,mean_full_pipeline_seconds,mean_improvement_over_previous_pct,mean_gap_to_unrestricted_pct
0,baseline,20,19,19,1.0000,46.1239,50.9227,0.0105,0.0000
1,densest_subgraph,20,20,20,0.1489,1.5374,6.3363,-0.0386,0.0512
2,highest_k_core,20,20,20,0.1723,3.2029,8.0017,-0.0041,0.0149
3,k_core_densest_union,20,20,20,0.1724,3.2078,8.0066,-0.0041,0.0149
4,max_p_cut,20,19,19,1.0000,49.9656,54.7644,0.0027,0.0078


In [16]:
display(wide_summary_df.head().round(4))

,instance,instance_id,n,p,best_known_cost,metaheuristic_best_cost,metaheuristic_gap_percent,metaheuristic_runtime_seconds,structure_extraction_runtime_seconds,perturbation_runtime_seconds,pre_reoptimization_seconds,previous_solution_perturbed_objective,previous_solution_degradation_percent,memory_size,top_solution_count,top_cost_cutoff,cooccurrence_edges,cooccurrence_total_weight,max_p_cut_fraction_cut,max_p_cut_best_facility_separation,k_core_max_level,k_core_candidate_count,k_core_candidate_fraction,k_core_best_set_recall,densest_subgraph_candidate_count,densest_subgraph_candidate_fraction,densest_subgraph_density,densest_subgraph_best_set_recall,k_core_densest_union_candidate_count,k_core_densest_union_candidate_fraction,k_core_densest_union_best_set_recall,perturbation_delta,perturbation_seed,perturbation_edge_count,perturbation_mean_multiplier,baseline__status,densest_subgraph__status,highest_k_core__status,k_core_densest_union__status,max_p_cut__status,baseline__candidate_facility_count,densest_subgraph__candidate_facility_count,highest_k_core__candidate_facility_count,k_core_densest_union__candidate_facility_count,max_p_cut__candidate_facility_count,baseline__candidate_facility_fraction,densest_subgraph__candidate_facility_fraction,highest_k_core__candidate_facility_fraction,k_core_densest_union__candidate_facility_fraction,max_p_cut__candidate_facility_fraction,baseline__objective_value,densest_subgraph__objective_value,highest_k_core__objective_value,k_core_densest_union__objective_value,max_p_cut__objective_value,baseline__improvement_over_previous_percent,densest_subgraph__improvement_over_previous_percent,highest_k_core__improvement_over_previous_percent,k_core_densest_union__improvement_over_previous_percent,max_p_cut__improvement_over_previous_percent,baseline__gap_to_unrestricted_percent,densest_subgraph__gap_to_unrestricted_percent,highest_k_core__gap_to_unrestricted_percent,k_core_densest_union__gap_to_unrestricted_percent,max_p_cut__gap_to_unrestricted_percent,baseline__model_build_seconds,densest_subgraph__model_build_seconds,highest_k_core__model_build_seconds,k_core_densest_union__model_build_seconds,max_p_cut__model_build_seconds,baseline__solve_seconds,densest_subgraph__solve_seconds,highest_k_core__solve_seconds,k_core_densest_union__solve_seconds,max_p_cut__solve_seconds,baseline__total_variant_seconds,densest_subgraph__total_variant_seconds,highest_k_core__total_variant_seconds,k_core_densest_union__total_variant_seconds,max_p_cut__total_variant_seconds,baseline__full_pipeline_seconds,densest_subgraph__full_pipeline_seconds,highest_k_core__full_pipeline_seconds,k_core_densest_union__full_pipeline_seconds,max_p_cut__full_pipeline_seconds
0,pmed1.txt,pmed1,100,5,5819.0,5819.0,0.0,0.0348,0.0206,0.0109,0.0664,5830,0.1890,131,14,5859.0,41,140.0,1.0,1.0000,6,10,0.10,1.0000,7,0.07,13.8571,1.0000,10,0.10,1.0000,0.1,42,198,0.9987,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,100,7,10,10,100,1.0,0.07,0.1,0.1,1.0,5830.0,5830.0,5830.0,5830.0,5830.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.236699,0.006328,0.010548,0.008876,0.092062,1.174018,0.051744,0.074485,0.099867,0.62263,1.410717,0.058073,0.085033,0.108743,0.714692,1.477148,0.124504,0.151464,0.175174,0.781123
1,pmed2.txt,pmed2,100,10,4093.0,4093.0,0.0,0.0681,0.0129,0.0102,0.0912,4087,-0.1466,121,13,4101.0,104,585.0,1.0,1.0000,11,13,0.13,1.0000,11,0.11,42.7273,1.0000,13,0.13,1.0000,0.1,43,193,0.9922,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,100,11,13,13,100,1.0,0.11,0.13,0.13,1.0,4087.0,4087.0,4087.0,4087.0,4087.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.080494,0.012508,0.011181,0.011525,0.081293,1.98557,0.076124,0.089718,0.091882,0.643714,2.066064,0.088632,0.100899,0.103406,0.725007,2.157264,0.179832,0.192098,0.194606,0.816207
2,pmed3.txt,pmed3,100,10,4250.0,4250.0,0.0,0.0611,0.0137,0.0102,0.0851,4237,-0.3059,127,13,4256.0,123,585.0,1.0,1.0000,11,14,0.14,1.0000,10,0.10,40.0000,0.9000,14,0.14,1.0000,0.1,44,198,0.9930,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,OPTIMAL,100,10,14,

In [17]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_df          .to_csv(RAW_RESULTS_CSV      , index=False)
    wide_summary_df     .to_csv(SUMMARY_RESULTS_CSV  , index=False)
    variant_aggregate_df.to_csv(VARIANT_AGGREGATE_CSV, index=False)

    print(f'Raw results saved to       : {RAW_RESULTS_CSV      }')
    print(f'Summary results saved to   : {SUMMARY_RESULTS_CSV  }')
    print(f'Variant aggregate saved to : {VARIANT_AGGREGATE_CSV}')

    if not failures_df.empty:
        failures_df.to_csv(FAILURES_CSV, index=False)

        print(f'Failures saved to        : {FAILURES_CSV}')

Raw results saved to       : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_interpretability_raw.csv
Summary results saved to   : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_interpretability_summary.csv
Variant aggregate saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_interpretability_variant_aggregate.csv
